# BayesFactor prior sensitivity on Rohwer Lo-SES

For each outcome (SAT, PPVT, Raven) and each PA predictor (n, s, ns, na, ss),
fits `BayesFactor::lmBF` against a varying Cauchy prior scale
r = 0.3, 0.5, sqrt(2)/2 (~0.707, the package default), 1.0.

This is the comparison that motivates the data-driven, empirical-Bayes prior
in `bootridge`: BF10 here visibly shifts with an arbitrary, user-chosen r,
whereas bootridge's ridge prior scale is tuned directly from the data via
.632 bootstrap prediction error.

In [ ]:
library(BayesFactor)
library(dplyr)
library(tidyr)
library(ggplot2)

dat <- read.csv("../data/rohwer_data.csv")
lo  <- dat[dat$SES == "Lo", ]
cat("Lo-SES sample size:", nrow(lo), "\n")

predictors <- c("n", "s", "ns", "na", "ss")
outcomes   <- c("SAT", "PPVT", "Raven")
r_scales   <- c(0.3, 0.5, sqrt(2) / 2, 1.0)


In [ ]:
results <- data.frame()

for (outcome in outcomes) {
  for (predictor in predictors) {
    for (r in r_scales) {
      f  <- as.formula(paste(outcome, "~", predictor))
      bf <- lmBF(f, data = lo, rscaleCont = r)
      logbf10 <- bf@bayesFactor$bf
      results <- rbind(results, data.frame(
        outcome   = outcome,
        predictor = predictor,
        r_scale   = r,
        logBF10   = logbf10,
        BF10      = exp(logbf10)
      ))
    }
  }
}

write.csv(results, "../output/bayesfactor_prior_sensitivity.csv", row.names = FALSE)
head(results)


In [ ]:
ggplot(results, aes(x = factor(round(r_scale, 3)), y = logBF10,
                     group = predictor, color = predictor)) +
  geom_line() +
  geom_point() +
  geom_hline(yintercept = 0, linetype = "dashed", color = "grey50") +
  facet_wrap(~ outcome, scales = "free_y") +
  labs(x = "Cauchy prior scale (r)", y = "log(BF10)",
       title = "Prior sensitivity: BayesFactor regression, Rohwer Lo-SES",
       subtitle = "Dashed line = BF10 of 1 (no preference)") +
  theme_minimal()

ggsave("../output/prior_sensitivity_plot.png", width = 9, height = 5)


**For the paper:** the predictor/outcome pairs where the qualitative
BF10 conclusion (e.g. crossing lnBF10 = 0, or crossing the "substantial
evidence" line at lnBF10 ~ 1.1 / BF10 ~ 3) flips across r = 0.3-1.0 are the
ones worth citing as concrete evidence of subjective-prior sensitivity —
pull those rows out of `bayesfactor_prior_sensitivity.csv` and cross-reference
against the same outcome/predictor's lnBF10 from `bootridge` in
`01_bootridge_lo_ses.ipynb`.